# MiniMax H3 Max Turbo

<p align="center">
  <img src="../../.github/images/banners/fal-minimax-h3-max-turbo.png" alt="MiniMax H3 Max Turbo: how-to clips from the product manuals, at Turbo speed" width="900" />
</p>

Fast text-to-video and image-to-video with first and last frames, prompt expansion, and native audio, from MiniMax (post-trained by fal) on Microsoft Foundry.

| | |
|---|---|
| Released | 2026-11-06 (preview) |
| Publisher | fal (model by MiniMax, post-trained by fal) |
| Capability | [Video Generation](../../3-primers/video-generation.md) |
| Model card | [Fal-MiniMax-H3-Max-Turbo](https://ai.azure.com/catalog/models/Fal-MiniMax-H3-Max-Turbo) |
| Announcement | [Release post](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/building-the-next-generation-of-multimodal-ai-experiences-with-fal-in-microsoft-/4570032) |
| Docs | [Fal models on Microsoft Foundry](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models) |
| Deployment | Global Standard, pay-as-you-go |
| Modes | Text-to-video, image-to-video (first and/or last frame). No reference-to-video (that's [H3 Max](../fal-minimax-h3-max/)) |
| Clip shape | 0.92–15 s (default 5; output can run ~0.7 s longer) · 480P, 768P, 1080P (refined from 768P) · 21:9 to 9:16 · audio always generated |
| Rate limit | 2 requests per minute on quota tiers 1–4 |
| Pricing | Not on the model card yet. Check your deployment in the Foundry portal; fal's direct price is on [fal.ai](https://fal.ai/models/minimax/h3-max-turbo/text-to-video) |
| Env vars | `MICROSOFT_FOUNDRY_ENDPOINT`, `AZURE_FAL_MINIMAX_H3_MAX_TURBO_DEPLOYMENT`, `DEMO_MODE` (optional: `AZURE_FAL_MINIMAX_H3_MAX_DEPLOYMENT`) |

**Read first:** the [Video Generation primer](../../3-primers/video-generation.md) explains the terms we use here (text-to-video, image-to-video, prompt expansion, seed) and when *not* to use generated video for instructions. Ten minutes there makes this notebook much easier.

**First time here?** Run [setup.ipynb](../../setup.ipynb) once to create your Microsoft Foundry project and `.env`.

> *"Can MiniMax H3 Max Turbo turn our product manuals into short how-to clips, quickly, and can we trust what they show?"*

Contoso Outdoors' support team wants a short clip for each setup step in the product manuals. Turbo is the fast tier of H3: same text and image modes as [H3 Max](../fal-minimax-h3-max/), built for speed. The manuals in the [Contoso Outdoors scenario](../../2-scenarios/contoso-retail/) are machine-generated and imperfect, which makes them a good test of a hard question: **does the video agree with the source?**

## What We Will Do

- Turn manual steps into clips, prompt by prompt, and time them
- Catch a clip that shows more than the manual promises, and check prompts against safety notes
- Animate the setup infographic from the earlier MAI workshop and compare it with the manual
- Compare Turbo with Max on one prompt, then package what we learn for a customer conversation

> Run each cell with **Shift + Enter**. Guess first: before each video, predict one thing it will get right and one it will get wrong.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Before you start

**Just reading?** Leave `DEMO_MODE=replay`. The notebook shows clips saved in `examples/` and makes no Azure calls.

**Running it live?** We need four things:

1. **fal enabled on the subscription.** An admin registers the **Fal.EnableDeploy** preview feature ([steps](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models#enable-fal-on-foundry)). It can take 30 minutes.
2. **A MiniMax H3 Max Turbo deployment** (Global Standard) in our Foundry project ([steps](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models#deploy-fal-models-from-the-foundry-portal)). Section 5 also uses **H3 Max** if it's deployed.
3. **A sign-in with a Foundry role** on the resource: run `az login` in a terminal. We use keyless Microsoft Entra ID tokens, never API keys.
4. **`.env` at the repo root** with `MICROSOFT_FOUNDRY_ENDPOINT`, `AZURE_FAL_MINIMAX_H3_MAX_TURBO_DEPLOYMENT`, and `DEMO_MODE=live`. See [sample.env](../../1-scripts/sample.env).

**Data handling.** fal hosts this model outside Microsoft. Prompts and images we send leave Microsoft, and Microsoft's product terms and DPA don't apply. We only send fictional Contoso data here.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "1-scripts" / "fal_video.py").exists())
load_dotenv(ROOT / ".env")

DEMO_MODE = None  # Set to "live" or "replay" to override .env
MODE = (DEMO_MODE or os.getenv("DEMO_MODE") or "replay").strip().lower()
ENDPOINT = os.getenv("MICROSOFT_FOUNDRY_ENDPOINT", "").strip()
DEPLOYMENT = os.getenv("AZURE_FAL_MINIMAX_H3_MAX_TURBO_DEPLOYMENT", "Fal-MiniMax-H3-Max-Turbo").strip()
MAX_DEPLOYMENT = os.getenv("AZURE_FAL_MINIMAX_H3_MAX_DEPLOYMENT", "").strip()
RPM = int(os.getenv("FAL_REQUESTS_PER_MINUTE", "2"))
PLAY_INLINE = os.getenv("PLAY_INLINE", "true").strip().lower() == "true"

if MODE not in ("replay", "live"):
    raise SystemExit(f'DEMO_MODE is "{MODE}". Use "replay" or "live".')

credential = None
if MODE == "live":
    if not ENDPOINT.startswith("https://"):
        raise SystemExit("MICROSOFT_FOUNDRY_ENDPOINT is missing from .env. Run setup.ipynb, or copy it from the project's Overview page.")
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    try:
        credential.get_token("https://cognitiveservices.azure.com/.default")
    except Exception as error:
        raise SystemExit(f"Couldn't get a Microsoft Entra token. Run `az login` in a terminal, then run this cell again.\n{error}")

print(f"Mode: {MODE} | Deployment: {DEPLOYMENT} | Max for section 5: {MAX_DEPLOYMENT or 'not set'} | {RPM} requests per minute")

The shared helper in [1-scripts/fal_video.py](../../1-scripts/fal_video.py) submits each job, polls until it's done, downloads the MP4, and saves a JSON record. `steps()` reads numbered steps from a product manual, so our prompts come straight from the source.

In [ ]:
import re

sys.path.insert(0, str(ROOT / "1-scripts"))
from fal_video import FalVideo, as_url, scoreboard

HERE = ROOT / "4-models" / "fal-minimax-h3-max-turbo"
DATA = ROOT / "2-scenarios" / "contoso-retail" / "data"
IMAGES = DATA / "generated" / "images"

fal = FalVideo(ENDPOINT, DEPLOYMENT, family="minimax/h3-max-turbo", here=HERE, mode=MODE,
               requests_per_minute=RPM, play_inline=PLAY_INLINE, credential=credential)
clips = []
SEED = 2026


def manual(slug: str) -> str:
    return (DATA / "manuals" / f"{slug}.md").read_text(encoding="utf-8")


def steps(slug: str, section: str = "Tent Setup") -> dict[str, str]:
    """{'Step 1: Selecting a Suitable Location': 'Find a level ... Remove any sharp ...', ...}"""
    block = manual(slug).split(f"### {section}", 1)[1].split("\n### ", 1)[0]
    found = re.findall(r"#### (Step \d+: [^\n]+)\n+((?:- [^\n]+\n?)+)", block)
    return {title: " ".join(b[2:].strip() for b in bullets.strip().splitlines()) for title, bullets in found}


SETUP = steps("trailmaster-x4-tent")
for title, text in SETUP.items():
    print(f"{title}\n   {text}")

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 1. How quickly can a manual step become a clip?

We wrap each step in the same shot template: a fixed look, the step's own words as the action, and quiet ambient sound. 480P and 5 seconds keep each draft quick. We make three steps, then look at the time each took.

> 🔮 **Guess first:** will "connect the tent poles according to their color codes" show color codes?

In [ ]:
TENT = "the TrailMaster X4, a four-person dome tent with a forest-green body and a burnt-orange rainfly"


def how_to(step_text: str) -> str:
    return (f"Instructional how-to video about {TENT}. Clear, well-lit meadow in daylight, eye-level camera, "
            f"hands and tent always in frame. Action: {step_text} Audio: quiet outdoor ambience, no music, no speech.")

picked = ["Step 1: Selecting a Suitable Location", "Step 3: Assembling the Tent Poles", "Step 5: Attaching the Rainfly (if applicable)"]
how_tos = [fal.run(f"01-howto-step-{title.split(':')[0][-1]}", "text-to-video", prompt=how_to(SETUP[title]), prompt_expansion_mode="disabled",
                   seed=SEED, duration=5, resolution="480P", aspect_ratio="16:9")
           for title in picked]
clips += how_tos
fal.show(*how_tos)

**What to look for:** step by step, does the clip show the **instruction**, or just "someone near a tent"? Look at the step 3 clip: color-coded poles, sleeves, or clips? Read the "made in" times: Turbo is built so a reviewer can watch drafts almost as fast as they arrive.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 2. Does the clip promise more than the manual?

This manual contradicts itself: its features say **"water-resistant construction to withstand light rain"**, its specs say **"Waterproof: Yes"**, and a review says it kept a family dry in **heavy** rain. A video model will happily show whichever claim we give it. We render both and compare.

In [ ]:
for line in manual("trailmaster-x4-tent").splitlines():
    if re.search(r"light rain|Waterproof\*\*|heavy rain", line, re.IGNORECASE):
        print("  ", line.strip()[:140])

claims = {
    "light-rain": f"{TENT} pitched in a meadow during a light drizzle; drops bead on the rainfly and roll off. Static wide shot. Audio: soft rain.",
    "storm": f"{TENT} pitched in a meadow during a violent thunderstorm with sideways rain; inside, a family stays perfectly dry. Audio: thunder, heavy rain.",
}
rain = [fal.run(f"02-claim-{name}", "text-to-video", prompt=prompt, prompt_expansion_mode="disabled",
                seed=SEED, duration=5, resolution="480P") for name, prompt in claims.items()]
clips += rain
fal.show(*rain)

**What to look for:** both clips look convincing, and that's the point. The model doesn't know which claim is true; it renders whatever we write. For product video, **the prompt is a product claim**. Ground it in the weakest supported statement (here, light rain) and have the product owner review every clip before it ships.

The same goes for safety. The EcoFire stove manual says it must never be used indoors or in enclosed spaces. A prompt like "cooking inside the tent on a rainy night" would produce a clip that teaches the wrong thing. A simple check of prompts against each manual's cautions catches the obvious cases:

In [ ]:
def cautions(slug: str) -> list[str]:
    return [line.strip("-0123456789. ").strip() for line in manual(slug).splitlines()
            if re.search(r"\b(do not|never|avoid)\b", line, re.IGNORECASE)]


draft_prompt = "A hiker cooks dinner on the EcoFire stove inside the closed tent on a rainy night."
for rule in cautions("ecofire-camping-stove"):
    if re.search(r"indoors|enclosed", rule, re.IGNORECASE) and re.search(r"inside|indoors|closed", draft_prompt, re.IGNORECASE):
        print(f"Blocked before generation. The prompt conflicts with the manual: {rule}")

A keyword check is a floor, not a guardrail; it shows *where* a review step belongs (before we spend a render), not how to build one. A production pipeline would ask a language model to compare the prompt with the manual's cautions.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 3. Can we animate the setup infographic?

The earlier MAI workshop made a flat-vector setup guide. With image-to-video, Turbo can animate it: a motion graphic instead of a photoreal scene, which suits instructions better. We ask for motion only.

In [ ]:
from IPython.display import Image as Picture, display

display(Picture(filename=str(IMAGES / "setup-guide.webp"), width=420))

guide = fal.run("03-animated-guide", "image-to-video", image_url=as_url(IMAGES / "setup-guide.webp"), prompt_expansion_mode="disabled",
                prompt="Each panel animates in turn from left to right: the parts slide into place and a small checkmark appears. Flat vector style, no new text. Audio: soft clicks.",
                seed=SEED, duration=8, resolution="768P")
clips.append(guide)
fal.show(guide, frames=8)

**What to look for:** two separate checks. First, **against the image**: does the flat style and any text survive the motion? Second, **against the manual**: the infographic came from an image model, so its steps may not match the manual's six. Generated assets inherit every upstream mistake; check each one against the source of truth, not against the previous asset.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 4. Can we show the before and after of a step?

Turbo supports first and last frames too. A how-to is a natural fit: the **before** state and the **after** state are known; only the motion between them is generated. We use the campsite pair from the workshop as "before and after attaching a new rainfly".

In [ ]:
display(Picture(filename=str(IMAGES / "campsite-before.webp"), width=280), Picture(filename=str(IMAGES / "campsite-after.webp"), width=280))

swap = fal.run("04-before-after", "image-to-video", image_url=as_url(IMAGES / "campsite-before.webp"),
               end_image_url=as_url(IMAGES / "campsite-after.webp"), prompt_expansion_mode="disabled",
               prompt="A camper drapes a new rainfly over the tent and clips it to each corner, then steps out of frame. Static camera. Audio: fabric rustle, clips snapping.",
               seed=SEED, duration=6, resolution="768P")
clips.append(swap)
fal.show(swap, frames=6)

**What to look for:** the end state should match the second image exactly; the motion in between is invented, so check it against **Step 5** in the manual ("attach the rainfly to the tent corners and secure it with the provided buckles or clips"). Fixed endpoints make a clip safer for instructions, but they don't make the middle correct.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 5. When is Turbo good enough?

Same prompt, same seed, same settings, two models. This section runs only if `AZURE_FAL_MINIMAX_H3_MAX_DEPLOYMENT` is set.

> 🔮 **Guess first:** how much faster will Turbo be? Will you be able to tell which clip is which without the labels?

In [ ]:
if MAX_DEPLOYMENT or MODE == "replay":
    max_model = FalVideo(ENDPOINT, MAX_DEPLOYMENT or "Fal-MiniMax-H3-Max", family="minimax/h3-max", here=HERE, mode=MODE,
                         requests_per_minute=RPM, play_inline=PLAY_INLINE, credential=credential)
    SAME = dict(prompt=how_to(SETUP["Step 4: Setting up the Tent Body"]), prompt_expansion_mode="disabled",
                seed=SEED, duration=5, resolution="768P", aspect_ratio="16:9")
    turbo_side = fal.run("05-compare-turbo", "text-to-video", **SAME)
    max_side = max_model.run("05-compare-max", "text-to-video", **SAME)
    clips += [turbo_side, max_side]
    fal.show(turbo_side, max_side)
else:
    print("Set AZURE_FAL_MINIMAX_H3_MAX_DEPLOYMENT in .env to run this comparison.")

**What to look for:** the "made in" times first, then the details that matter for **this** job: hands, poles, and fabric behaving correctly. For a support library where every clip is reviewed anyway, Turbo's speed often wins; for a hero campaign shot, Max's extra fidelity, and its reference-to-video mode, is worth the wait.

### 🔧 Break it on purpose

H3 clips must be at least **0.92 seconds**. What does a half-second request return?

In [ ]:
if MODE == "live":
    try:
        fal.run("break-half-second", "text-to-video", prompt=how_to(SETUP["Step 1: Selecting a Suitable Location"]),
                prompt_expansion_mode="disabled", duration=0.5, resolution="480P")
    except RuntimeError as error:
        print(error)
else:
    print("Replay mode: in live mode the service answers 422 for `duration`, which must be at least 0.92 seconds.")

Unlike Kling and LTX, H3 takes duration as a **number**, any value from 0.92 to 15, not a fixed list. That suits clips cut to an exact beat, like a 2.5-second bumper, but remember the delivered clip can run up to ~0.7 s longer.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 6. What did each clip cost us in time?

### Scoreboard

Every clip we made, from its saved record: how long the job took end to end, the duration we asked for vs the one we got, and the output's shape and size.

In [ ]:
scoreboard(clips)

**What to look for:** Turbo's times at 480P vs 768P, the Turbo-vs-Max gap from section 5, and the asked-vs-delivered duration column. A full six-step manual at 2 requests per minute is about 3 minutes of submits plus render time, per product. For pricing, use the deployment's page in the Foundry portal, not estimates.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Teach it to a customer

### What it does, and why it matters

- **The fast tier of MiniMax H3**: text-to-video and image-to-video with native audio, built for speed.
- **First and last frames** (or last only): fix the before and after, generate the motion.
- **Flexible duration** (0.92–15 s, any value) and six aspect ratios, from 21:9 to 9:16.
- **Readable prompt expansion and a seed**, the same as H3 Max.

### A prompt recipe that works

Template the **look** once (setting, camera, light, audio), and fill the **action** from the source document, word for word. Disable prompt expansion so the source text isn't rewritten.

### Honest answers to common questions

- **Can it make our instructional videos?** It can make *illustrations* of steps. A generated clip can show a wrong or unsafe step convincingly; an expert must review each one.
- **Is it production-ready?** It's in public preview on Foundry. Validate quality and terms before production.
- **Where does our data go?** To fal, outside Microsoft. Microsoft's product terms, DPA, and certifications don't apply. Review [fal's terms](https://fal.ai/legal) with the customer.
- **Turbo or Max?** Turbo for volume and drafts; [Max](../fal-minimax-h3-max/) for references (images, video, audio) and final assets.

### When something goes wrong

- **The clip shows "someone near a tent", not the step** → name one concrete action with a start and end, and keep hands in frame.
- **The clip contradicts the manual** → the prompt carried the wrong claim; ground it in the source and review.
- **422 on `duration`** → under 0.92 or over 15 seconds.
- **We need our real product in shot** → use image-to-video, or H3 Max's reference-to-video.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Your Turn to Explore

- **Takedown, too.** Run `steps("trailmaster-x4-tent", "Tent Takedown and Storage")` and make a clip for each takedown step.
- **Another product.** Pick a backpack or sleeping bag manual and write a `steps()` variant for its format.
- **Exact beats.** Make a 2.5-second "Step 1" bumper and a 7.5-second step clip. Check the delivered durations.

In [ ]:
# Your experiment here. Try: fal.run("my-clip", "text-to-video", prompt=how_to("..."), duration=5, resolution="480P")

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## What We Did

- ✅ Turned manual steps into clips, prompt by prompt, and timed them
- ✅ Caught a clip that promised more than the manual, and checked prompts against safety notes
- ✅ Animated the setup infographic and compared it with the manual
- ✅ Compared Turbo with Max on one prompt and packaged the findings for a customer conversation

➡️ **Next:** [MiniMax H3 Max](../fal-minimax-h3-max/) adds reference-to-video and a pinned soundtrack.

Stuck? [Open an issue](https://github.com/microsoft-foundry/model-releases/issues/new) with the failing cell's output.

**Learn more:** [Fal models on Microsoft Foundry](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models) · [Video Generation primer](../../3-primers/video-generation.md) · [MiniMax H3 Max Turbo API on fal](https://fal.ai/models/minimax/h3-max-turbo/text-to-video/api)